# glproc MoE verification + thread scaling â€” Veritas Prima Fix 2

Two questions this notebook answers, both of which the dev laptop (i3-1115G4, 2 physical / 4 logical cores, 8 GB) **cannot**:

1. **Is the MoE compute path correct at Qwen3's real scale?** 128 experts, top-8, Qwen3-30B-A3B's actual dims â€” verified against a naive reference, and against the Q8_0 integer-dot kernels (the laptop tests only exercise the f32 fallback).
2. **Does thread count actually scale?** Fix 1's physical-core hypothesis *lost by 23%* on 2 cores. Colab gives more cores â€” so we sweep `GLPROC_THREADS` and find where it really tops out, instead of guessing.

**Runtime: CPU** (Runtime â†’ Change runtime type â†’ CPU). Internet on. ~4 min, no model download â€” everything is synthetic, so nothing here depends on a 17 GB Qwen3-MoE file that won't fit anywhere.

Each cell prints ONE clear result. Read top to bottom.

## 1 Â· Machine + Rust toolchain

Print the CPU we actually got. Colab hands out different hosts; thread-scaling numbers are meaningless without knowing the core count they came from.

In [ ]:
import subprocess, os, re

cpuinfo = open('/proc/cpuinfo').read()
model = re.search(r'model name\s*:\s*(.+)', cpuinfo)
logical = cpuinfo.count('processor\t:')
cores_field = re.search(r'cpu cores\s*:\s*(\d+)', cpuinfo)
physical = int(cores_field.group(1)) if cores_field else logical
flags = re.search(r'^flags\s*:\s*(.+)$', cpuinfo, re.M).group(1).split()
isa = [f for f in ('avx2', 'fma', 'f16c', 'avx512f', 'avx512_vnni') if f in flags]
mem_gb = int(re.search(r'MemTotal:\s*(\d+)', open('/proc/meminfo').read()).group(1)) / 1e6

print(f"CPU        : {model.group(1).strip() if model else '?'}")
print(f"physical   : {physical}")
print(f"logical    : {logical}   (SMT {'on' if logical > physical else 'off'})")
print(f"ISA        : {' '.join(isa)}")
print(f"RAM        : {mem_gb:.1f} GB")
print()
print("NOTE: glproc's SimdStrategy rejects AVX-512 on <=8-logical-core parts")
print("      (mobile downclock heuristic). On a big Colab host it may take it.")

# Rust toolchain (Colab has none by default).
if subprocess.run(['which', 'cargo'], capture_output=True).returncode != 0:
    print('\ninstalling rust...')
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable >/dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version

## 2 Â· Fetch the repo â€” stale-proof

Fetch the GitHub URL **directly** and hard-reset to it. The r256 notebook got burned by `git pull` resolving to the wrong remote and reporting *"Already up to date"* while sitting on stale code, so we don't trust the remote config at all â€” and we **prove `moe.rs` is on disk** before building.

In [ ]:
REPO = 'https://github.com/JinXSuperSolo/gwenland-ai'
BRANCH = 'main'

import os, subprocess
if not os.path.isdir('/content/gwenland/.git'):
    !rm -rf /content/gwenland
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/gwenland
os.chdir('/content/gwenland')

# Never trust 'origin' â€” fetch the URL itself, hard-reset to what came back.
!git fetch -q --depth 1 {REPO} {BRANCH}
!git reset -q --hard FETCH_HEAD
!git log --oneline -3

# Prove the code under test is actually here. A missing moe.rs means the
# fetch silently resolved to something old and every result below is a lie.
assert os.path.exists('glproc/src/moe.rs'), 'moe.rs MISSING â€” stale checkout, stop.'
src = open('glproc/src/moe.rs').read()
assert 'MAX_TOP_K' in src and 'fn softmax_top_k' in src, 'moe.rs present but wrong content'
print(f"\nOK: moe.rs on disk, {len(src.splitlines())} lines")

## 3 Â· Existing test suite must stay green

Before trusting any new number: does the whole crate still pass on a machine that isn't the dev laptop? This runs the 57 lib tests (46 pre-existing + 11 MoE) and the 14 kernel-parity tests.

`--release` matters â€” several kernels only take their SIMD path when optimized, and `QuantizedActivation::quantize` only `debug_assert`s its bounds, so a release run is the one that would actually corrupt memory if the scratch sizing were wrong.

In [ ]:
!cargo test -q -p glproc --release 2>&1 | grep -vE 'warning: profiles|^package:|^workspace:' | tail -25

## 4 Â· MoE correctness at Qwen3's REAL scale â€” the gate

The in-crate tests use small synthetic dims (16â€“128 experts, hidden 32) so they run in milliseconds. This is the test they can't be: **Qwen3-30B-A3B's actual per-layer shape**, and crucially with **Q8_0 quantized experts**, which drives `par_matvec_swiglu` / `par_matmul_swiglu` and the integer-dot kernels â€” the code path a real model takes and the laptop tests never reach.

| | Qwen3-30B-A3B |
|---|---|
| experts | 128 |
| top_k | **8** (not 2 â€” that's Mixtral) |
| hidden | 2048 |
| expert_ffn | 768 |

Note `expert_ffn (768) < hidden (2048)`. That inequality is why the shared-`acts` out-of-bounds bug would have stayed *invisible* on this model â€” the test below deliberately also runs a `ffn > hidden` shape, which is the one that would have corrupted memory in release.

This is written as a Rust integration test (not a bench) so it runs under the real kernels with real dispatch.

In [ ]:
%%writefile glproc/tests/moe_colab.rs
//! Colab-only MoE verification at Qwen3-30B-A3B scale, on Q8_0 experts.
//! Too slow for the default suite (128 experts x 2048x768); run explicitly.

use glproc::kernels::bridge::QuantFormat;
use glproc::kernels::dequant::q8_0;
use glproc::model::{GateUp, WeightMatrix};
use glproc::moe::{ExpertWeights, MoEConfig, MoELayer};
use glproc::simd_strategy::SimdStrategy;
use glproc::threading::ThreadPool;

/// Deterministic pseudo-random f32 in [-1, 1). No rand dep (crate has none).
fn prng(seed: &mut u64) -> f32 {
    *seed = seed
        .wrapping_mul(6364136223846793005)
        .wrapping_add(1442695040888963407);
    ((*seed >> 33) as f32 / (1u64 << 31) as f32) - 1.0
}

fn rand_vec(n: usize, seed: &mut u64) -> Vec<f32> {
    (0..n).map(|_| prng(seed)).collect()
}

/// Build an MoE layer whose experts are Q8_0, in the row-interleaved
/// GateUp::FusedQuant layout the loader produces â€” i.e. the real decode path.
/// Returns the layer plus the f32 weights it was quantized FROM, so the
/// reference can be computed without re-dequantizing.
fn build_q8_layer(
    ne: usize,
    k: usize,
    h: usize,
    f: usize,
    seed: &mut u64,
) -> (MoELayer, Vec<(Vec<f32>, Vec<f32>, Vec<f32>)>) {
    let router = rand_vec(ne * h, seed);
    let mut experts = Vec::with_capacity(ne);
    let mut f32_ref = Vec::with_capacity(ne);

    for _ in 0..ne {
        let gate = rand_vec(f * h, seed);
        let up = rand_vec(f * h, seed);
        let down = rand_vec(h * f, seed);

        // Quantize, then dequantize back to f32 â€” the reference must be
        // computed from the ROUNDED weights, else we'd be measuring
        // quantization error and calling it a routing bug.
        let gq = q8_0::scalar::quantize(&gate);
        let uq = q8_0::scalar::quantize(&up);
        let dq = q8_0::scalar::quantize(&down);

        // Row-interleave gate/up exactly as loader::fuse_gate_up does:
        // [gate row 0][up row 0][gate row 1]...
        let row_bytes = gq.len() / f;
        let mut packed = Vec::with_capacity(gq.len() + uq.len());
        for o in 0..f {
            packed.extend_from_slice(&gq[o * row_bytes..(o + 1) * row_bytes]);
            packed.extend_from_slice(&uq[o * row_bytes..(o + 1) * row_bytes]);
        }

        f32_ref.push((
            q8_0::scalar::run(&gq),
            q8_0::scalar::run(&uq),
            q8_0::scalar::run(&dq),
        ));
        experts.push(ExpertWeights {
            gate_up: GateUp::FusedQuant(QuantFormat::Q8_0, packed),
            w_down: WeightMatrix::Quant(QuantFormat::Q8_0, dq),
        });
    }

    let layer = MoELayer {
        config: MoEConfig {
            num_experts: ne,
            num_experts_per_tok: k,
            expert_ffn_size: f,
            hidden_size: h,
            norm_topk_prob: true,
        },
        router,
        experts,
    };
    (layer, f32_ref)
}

/// Naive MoE, straight from the definition. Uses f32::exp deliberately â€”
/// sharing fast_exp with the code under test would make them agree by
/// construction and stop testing the approximation.
fn reference(
    layer: &MoELayer,
    w: &[(Vec<f32>, Vec<f32>, Vec<f32>)],
    input: &[f32],
    n_tokens: usize,
) -> Vec<f32> {
    let c = &layer.config;
    let (h, f, ne, k) = (
        c.hidden_size,
        c.expert_ffn_size,
        c.num_experts,
        c.num_experts_per_tok,
    );
    let mut out = vec![0f32; n_tokens * h];

    for t in 0..n_tokens {
        let x = &input[t * h..(t + 1) * h];

        // Router -> softmax over ALL experts -> top-k -> renormalize.
        // This order is Qwen3's. Selecting before the softmax gives
        // different weights, so it is part of what we are verifying.
        let scores: Vec<f32> = (0..ne)
            .map(|e| (0..h).map(|i| layer.router[e * h + i] * x[i]).sum())
            .collect();
        let max = scores.iter().copied().fold(f32::NEG_INFINITY, f32::max);
        let exps: Vec<f32> = scores.iter().map(|s| (s - max).exp()).collect();
        let sum: f32 = exps.iter().sum();
        let probs: Vec<f32> = exps.iter().map(|e| e / sum).collect();

        let mut idx: Vec<usize> = (0..ne).collect();
        idx.sort_by(|&a, &b| probs[b].partial_cmp(&probs[a]).unwrap().then(a.cmp(&b)));
        let top = &idx[..k];
        let mass: f32 = top.iter().map(|&e| probs[e]).sum();

        for &e in top {
            let wt = probs[e] / mass;
            let (gw, uw, dw) = &w[e];
            let mut act = vec![0f32; f];
            for o in 0..f {
                let g: f32 = (0..h).map(|i| gw[o * h + i] * x[i]).sum();
                let u: f32 = (0..h).map(|i| uw[o * h + i] * x[i]).sum();
                act[o] = g / (1.0 + (-g).exp()) * u;
            }
            for o in 0..h {
                let d: f32 = (0..f).map(|i| dw[o * f + i] * act[i]).sum();
                out[t * h + o] += wt * d;
            }
        }
    }
    out
}

fn check(name: &str, ne: usize, k: usize, h: usize, f: usize, n_tokens: usize) {
    let mut seed = 0xC0FFEEu64;
    let (layer, wref) = build_q8_layer(ne, k, h, f, &mut seed);
    let input = rand_vec(n_tokens * h, &mut seed);
    let pool = ThreadPool::new(num_cpus::get());
    let strategy = SimdStrategy::detect();

    let mut got = vec![0f32; n_tokens * h];
    let routing = layer.forward(&input, &mut got, n_tokens, &pool, strategy);
    let want = reference(&layer, &wref, &input, n_tokens);

    // Relative tolerance: forward() and the reference sum the same products
    // in different orders (threaded/batched vs sequential), and the SwiGLU
    // goes through fast_exp on one side and libm exp on the other.
    let mut worst = 0f32;
    for (g, w) in got.iter().zip(&want) {
        let rel = (g - w).abs() / w.abs().max(1.0);
        worst = worst.max(rel);
    }

    let load = &routing.expert_load;
    let active = load.iter().filter(|&&c| c > 0).count();
    let (mn, mx) = (
        load.iter().copied().min().unwrap(),
        load.iter().copied().max().unwrap(),
    );
    println!(
        "{name:<26} tokens={n_tokens:<4} worst_rel_err={worst:.2e}  \
         experts_touched={active}/{ne}  load min/max={mn}/{mx}"
    );

    assert_eq!(
        load.iter().sum::<usize>(),
        n_tokens * k,
        "{name}: every token must be routed to exactly top_k experts"
    );
    // 2e-3 relative: Q8_0 is ~7 bits of mantissa, and error compounds across
    // three quantized projections plus the fast_exp approximation. A routing
    // or dispatch BUG shows up as 1e-1+, not 1e-3 â€” this bound separates them.
    assert!(worst < 2e-3, "{name}: worst relative error {worst:.3e} â€” too large");
}

#[test]
fn moe_q8_qwen3_30b_a3b_shape() {
    // The real thing: Qwen3-30B-A3B's per-layer MoE block.
    println!();
    check("qwen3-30b-a3b decode", 128, 8, 2048, 768, 1);
    check("qwen3-30b-a3b prefill", 128, 8, 2048, 768, 32);
}

#[test]
fn moe_q8_ffn_wider_than_hidden() {
    // The shape Qwen3 does NOT have: expert_ffn > hidden. This is the case
    // where a shared-acts buffer sized to `hidden` writes out of bounds â€”
    // silently, in release, since quantize() only debug_asserts the fit.
    // If the scratch sizing regresses to min instead of max, this corrupts.
    println!();
    check("ffn>hidden (OOB guard)", 16, 4, 256, 1024, 8);
}

#[test]
fn moe_top_k_is_not_hardcoded() {
    // Mixtral (top-2 of 8) and Qwen3 (top-8 of 128) must both work. A
    // hardcoded top_k passes one and fails the other.
    println!();
    check("mixtral-ish top-2/8", 8, 2, 512, 256, 16);
    check("qwen3-ish top-8/128", 128, 8, 512, 256, 16);
}

In [ ]:
# num_cpus is a glproc dep, but integration tests need it declared for themselves.
import re
ct = open('glproc/Cargo.toml').read()
if '[dev-dependencies]' not in ct:
    ct += '\n[dev-dependencies]\nnum_cpus = "1.16"\n'
    open('glproc/Cargo.toml', 'w').write(ct)

!cargo test -q -p glproc --release --test moe_colab -- --nocapture --test-threads=1 2>&1 \
  | grep -vE 'warning: profiles|^package:|^workspace:|Compiling|Finished'

**How to read this.** `worst_rel_err` around `1e-3` is Q8_0 quantization noise compounding through three projections plus `fast_exp` â€” expected and fine. A **routing or dispatch bug reads as `1e-1` or worse**, not `1e-3`; the two failure modes are orders of magnitude apart, which is why the assert sits at `2e-3`.

`experts_touched` is the load-balance signal. At 1 token Ã— top-8 it must be exactly 8 of 128 â€” if it says 128, the skip-empty path is broken and you're streaming 16Ã— the weights you should be, which is the entire MoE performance argument gone.

## 5 Â· Thread scaling â€” settle Fix 1 on real hardware

On the 2-core dev laptop, sizing the pool to *physical* cores **lost by 23%** (11.0 â†’ 8.5 tok/s): decode sits at only ~69% of the DRAM read ceiling, so the core still has idle issue slots that an SMT sibling fills. That's a 2-core result, though. Colab has more cores and a different memory system, so the honest thing is to sweep rather than extrapolate.

This benches the **MoE layer itself** (not full decode â€” no MoE GGUF exists to load), which is the workload we actually care about.

In [ ]:
%%writefile glproc/benches/moe_threads.rs
//! Thread-scaling sweep for the MoE layer. Not a criterion bench (crate has
//! zero external deps) â€” a plain binary that times pool sizes directly.

use std::time::Instant;

use glproc::kernels::bridge::QuantFormat;
use glproc::kernels::dequant::q8_0;
use glproc::model::{GateUp, WeightMatrix};
use glproc::moe::{ExpertWeights, MoEConfig, MoELayer};
use glproc::simd_strategy::SimdStrategy;
use glproc::threading::ThreadPool;

fn prng(seed: &mut u64) -> f32 {
    *seed = seed
        .wrapping_mul(6364136223846793005)
        .wrapping_add(1442695040888963407);
    ((*seed >> 33) as f32 / (1u64 << 31) as f32) - 1.0
}

fn main() {
    // Qwen3-30B-A3B's MoE block. 128 experts x (2*768*2048 + 2048*768) Q8_0
    // ~= 400 MB of expert weights â€” big enough that this is a real memory
    // workload, not an L2-resident toy.
    let (ne, k, h, f) = (128usize, 8usize, 2048usize, 768usize);
    let mut seed = 0xBEEFu64;

    eprintln!("building {ne} Q8_0 experts (~400 MB)...");
    let router: Vec<f32> = (0..ne * h).map(|_| prng(&mut seed)).collect();
    let experts: Vec<ExpertWeights> = (0..ne)
        .map(|_| {
            let gate: Vec<f32> = (0..f * h).map(|_| prng(&mut seed)).collect();
            let up: Vec<f32> = (0..f * h).map(|_| prng(&mut seed)).collect();
            let down: Vec<f32> = (0..h * f).map(|_| prng(&mut seed)).collect();
            let (gq, uq) = (q8_0::scalar::quantize(&gate), q8_0::scalar::quantize(&up));
            let row_bytes = gq.len() / f;
            let mut packed = Vec::with_capacity(gq.len() + uq.len());
            for o in 0..f {
                packed.extend_from_slice(&gq[o * row_bytes..(o + 1) * row_bytes]);
                packed.extend_from_slice(&uq[o * row_bytes..(o + 1) * row_bytes]);
            }
            ExpertWeights {
                gate_up: GateUp::FusedQuant(QuantFormat::Q8_0, packed),
                w_down: WeightMatrix::Quant(QuantFormat::Q8_0, q8_0::scalar::quantize(&down)),
            }
        })
        .collect();

    let layer = MoELayer {
        config: MoEConfig {
            num_experts: ne,
            num_experts_per_tok: k,
            expert_ffn_size: f,
            hidden_size: h,
            norm_topk_prob: true,
        },
        router,
        experts,
    };
    let strategy = SimdStrategy::detect();
    eprintln!("simd: {strategy:?}\n");

    let logical = num_cpus::get();
    println!("{:<8} {:>12} {:>12} {:>10}", "threads", "decode ms", "prefill32 ms", "speedup");
    println!("{}", "-".repeat(46));

    let mut base_decode = 0f64;
    for nt in 1..=logical {
        let pool = ThreadPool::new(nt);

        // decode: 1 token, top-8 experts -> matvec path
        let input: Vec<f32> = (0..h).map(|_| prng(&mut seed)).collect();
        let mut out = vec![0f32; h];
        for _ in 0..3 {
            layer.forward(&input, &mut out, 1, &pool, strategy); // warm
        }
        let iters = 20;
        let t = Instant::now();
        for _ in 0..iters {
            layer.forward(&input, &mut out, 1, &pool, strategy);
        }
        let decode_ms = t.elapsed().as_secs_f64() * 1e3 / iters as f64;

        // prefill: 32 tokens -> experts batched, matmul path
        let binput: Vec<f32> = (0..32 * h).map(|_| prng(&mut seed)).collect();
        let mut bout = vec![0f32; 32 * h];
        for _ in 0..2 {
            layer.forward(&binput, &mut bout, 32, &pool, strategy); // warm
        }
        let iters = 5;
        let t = Instant::now();
        for _ in 0..iters {
            layer.forward(&binput, &mut bout, 32, &pool, strategy);
        }
        let prefill_ms = t.elapsed().as_secs_f64() * 1e3 / iters as f64;

        if nt == 1 {
            base_decode = decode_ms;
        }
        println!(
            "{nt:<8} {decode_ms:>12.2} {prefill_ms:>12.2} {:>9.2}x",
            base_decode / decode_ms
        );
    }

    println!("\nphysical cores: {}", glproc::topology::physical_core_count());
    println!("logical threads: {logical}");
    println!("\nIf decode keeps improving past the physical-core count, SMT is");
    println!("filling issue slots and pool size should track LOGICAL threads.");
    println!("If it plateaus or regresses at physical, Fix 1 was right here and");
    println!("the i3 result was a 2-core artifact.");
}

In [ ]:
# Register the bench as a plain binary target (no criterion â€” zero-dep crate).
ct = open('glproc/Cargo.toml').read()
if 'moe_threads' not in ct:
    ct += '\n[[bench]]\nname = "moe_threads"\nharness = false\n'
    open('glproc/Cargo.toml', 'w').write(ct)

!cargo bench -q -p glproc --bench moe_threads 2>&1 \
  | grep -vE 'warning: profiles|^package:|^workspace:'

## 6 Â· Verdict

Fill this in from the cells above â€” the notebook exists to produce these two answers, so don't leave without them.

**MoE correctness (cell 4):** all three tests green, worst relative error ~`1e-3`, and `experts_touched = 8/128` on the single-token decode case? Then the compute path is right at Qwen3's real scale on the real quantized kernels, and the remaining risk is entirely in the **GGUF `_exps` loader** â€” which is not written, because the expert-stacking layout cannot be verified without a real Qwen3-MoE file.

**Thread scaling (cell 5):** compare the decode column's knee against the printed physical-core count.
- Still improving past physical â†’ SMT is filling issue slots, `N_THREADS` should track **logical**. Matches the i3 result; Fix 1 stays rejected, and the current code is already correct.
- Plateaus/regresses at physical â†’ the i3's 23% loss was a 2-core artifact, and pool sizing wants to be **topology-aware** rather than a constant. `topology::physical_core_count()` is already in the tree, unused, ready to wire up.

Either way it's a measurement, not a guess â€” which is the whole point, given Fix 1's plausible-sounding hypothesis turned out to be backwards.